# ColQwen2 visual retrieval arm — ViDoRe V3 physics

Chuẩn bị theo `docs/phan_tich_first_principles.md` §3-4 (mục 3 trong bảng ưu tiên). Không thành
phần nào trong pipeline hiện tại là một mô hình được huấn luyện để xếp hạng; đây là lần đầu chạy
một **document-VLM thật** (không phải CLIP tự nhiên-ảnh) trên corpus này.

**Mục tiêu đo cụ thể:** 473 trang gold (21.7% tổng số) hiện bị trượt hẳn (rank ≥ 100) và có
`fig_ratio` cao gấp 2.4 lần nhóm tìm thấy được — visual có tìm lại được chúng không?

**Trước khi chạy:**
1. `Runtime → Change runtime type → GPU` (T4 free tier là đủ; L4/A100 nhanh hơn).
2. Repo là **private**, nên notebook không `git clone` — thay vào đó build một file zip ở máy
   local (đã chứa cả code lẫn data, không cần token GitHub) rồi upload thẳng ở Bước 1:

   ```bash
   cd AXIOM_DE-RD
   git archive --format=zip -o /tmp/physics_colab_bundle.zip HEAD
   zip -r /tmp/physics_colab_bundle.zip \
     data/raw/benchmarks/vidore_v3_physics \
     data/benchmark/vidore_v3/physics \
     data/benchmark/vidore_v3/results/physics_served_pool.json
   ```
   `git archive` lấy đúng các file đã track trong git (code, ~2.5MB) tại `HEAD` hiện tại; lệnh
   `zip -r` sau đó thêm data gitignored (42 PDF + 3 parquet + served_pool.json, ~102MB) vào cùng
   file. Tổng ~105MB, một lần upload duy nhất.
3. Toàn bộ notebook **không gọi OpenRouter/Voyage** — chỉ tải model công khai từ HuggingFace và
   chạy suy luận cục bộ. Không tốn ngân sách API.

**Kết quả xuất ra:** 3 file nhỏ (`physics_colqwen_scores.npy`, `..._keys.json`, `..._qids.json`,
tổng cộng ~2-3MB) -- tải về máy local để chạy `research/experiments/physics_colqwen_eval.py`,
script này sẽ tính NDCG@10, fusion với arm text, và độ phủ trên đúng 473 trang bị trượt.

In [ ]:
!nvidia-smi

## 1. Upload bundle (code + data) và cài đặt

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

uploaded = files.upload()  # chon physics_colab_bundle.zip build o Buoc chuan bi ben tren
zip_name = next(iter(uploaded))

REPO = Path("/content/AXIOM_DE-RD")
REPO.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(REPO)

pdfs = list((REPO / "data/raw/benchmarks/vidore_v3_physics").glob("*.pdf"))
pool = REPO / "data/benchmark/vidore_v3/results/physics_served_pool.json"
pyproject = REPO / "pyproject.toml"
assert len(pdfs) == 42, f"expected 42 PDFs, found {len(pdfs)}"
assert pool.is_file(), "physics_served_pool.json missing after unzip"
assert pyproject.is_file(), "pyproject.toml missing -- did git archive run at repo root?"
print(f"OK: {len(pdfs)} PDFs, served pool present, package layout present")


In [ ]:
%cd /content/AXIOM_DE-RD
%pip install -q -e "."
%pip install -q "colpali-engine>=0.3.18" pymupdf
# Colab preinstalls torchao 0.10.0; peft (colpali-engine's LoRA dep) requires
# >=0.16.0 and raises ImportError on model load otherwise. Upgrade, then
# Runtime -> Restart session before continuing (files in /content survive a
# session restart, only a full disconnect wipes them -- no need to re-upload).
%pip install -q -U "torchao>=0.16.0"
print("torchao upgraded -- now Runtime > Restart session, then continue from the next cell.")


## 2. GPU profile

Chọn dtype theo GPU: T4 (Turing) không hỗ trợ bfloat16 native -> dùng float16. L4/A100/H100
(Ampere+) dùng bfloat16. Cùng quy ước với `Chandra_serving_de.ipynb`.

In [ ]:
import torch

gpu = torch.cuda.get_device_properties(0)
gpu_name = gpu.name
gpu_memory_gib = gpu.total_memory / 1024**3
print(f"GPU={gpu_name}, VRAM={gpu_memory_gib:.1f} GiB")

if any(x in gpu_name for x in ("A100", "H100", "L4", "L40")):
    DTYPE = torch.bfloat16
else:
    DTYPE = torch.float16  # T4 and anything else without native bf16
print(f"dtype = {DTYPE}")


## 3. Render page images từ PDF gốc

Dùng lại `research/experiments/render_page_images.py` đã có sẵn trong repo (đã dùng cho arm CLIP).

In [ ]:
%cd /content/AXIOM_DE-RD
!python research/experiments/render_page_images.py --dpi 144


In [ ]:
from pathlib import Path
images = sorted(Path("/content/AXIOM_DE-RD/data/work/vidore_physics_page_images").glob("*.png"))
print(f"{len(images)} page images rendered")
assert len(images) == 1674, f"expected 1674, got {len(images)}"


## 4. Load ColQwen2

`vidore/colqwen2-v1.0` -- Qwen2-VL-2B backbone, late-interaction (multi-vector), 89.3 NDCG@5 trên
ViDoRe v1 leaderboard. ~4.4GB tải về, ~5GB VRAM khi chạy fp16/bf16.

In [ ]:
from colpali_engine.models import ColQwen2, ColQwen2Processor
from transformers.utils.import_utils import is_flash_attn_2_available

MODEL_NAME = "vidore/colqwen2-v1.0"

model = ColQwen2.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="cuda:0",
    attn_implementation="flash_attention_2" if is_flash_attn_2_available() else None,
).eval()
processor = ColQwen2Processor.from_pretrained(MODEL_NAME)
print("model loaded")


## 5. Encode 1,674 trang -- có checkpoint, resumable

Mỗi ảnh encode riêng lẻ (batch=1): tránh mọi rủi ro về việc phải tự cắt padding ra khỏi
multi-vector embedding khi gộp batch (thư viện có hỗ trợ batch ảnh khác kích thước, nhưng cách an
toàn nhất để không làm hỏng MaxSim là không có padding ngay từ đầu). Lưu checkpoint mỗi 100 ảnh
đề phòng Colab ngắt phiên.

In [ ]:
import time
from PIL import Image

sample = images[:20]
t0 = time.time()
for img_path in sample:
    batch = processor.process_images([Image.open(img_path)]).to(model.device)
    with torch.no_grad():
        _ = model(**batch)
elapsed = time.time() - t0
per_image = elapsed / len(sample)
print(f"timing probe: {per_image:.2f}s/image -> ~{per_image * len(images) / 60:.1f} phut cho {len(images)} anh")


In [ ]:
import pickle

CKPT = Path("/content/colqwen_page_embeddings.pkl")
page_embeddings = {}  # unit_id -> CPU float16 tensor (seq_len, 128)
if CKPT.exists():
    page_embeddings = pickle.load(open(CKPT, "rb"))
    print(f"resuming: {len(page_embeddings)}/{len(images)} da encode")

for i, img_path in enumerate(images):
    key = img_path.stem.replace("__", "::")
    if key in page_embeddings:
        continue
    batch = processor.process_images([Image.open(img_path)]).to(model.device)
    with torch.no_grad():
        emb = model(**batch)[0].to(torch.float16).cpu()
    page_embeddings[key] = emb
    if (i + 1) % 100 == 0:
        pickle.dump(page_embeddings, open(CKPT, "wb"))
        print(f"  {i + 1}/{len(images)}", flush=True)

pickle.dump(page_embeddings, open(CKPT, "wb"))
print(f"done: {len(page_embeddings)} page embeddings")


## 6. Encode 302 câu truy vấn + tính ma trận điểm số

In [ ]:
import sys
sys.path.insert(0, "/content/AXIOM_DE-RD")
from src.evaluation.benchmarks import load

bench = load("vidore_v3", subset="physics", language="french")
qrels = bench.qrels()
questions = [q for q in bench.questions() if qrels.get(q.qid)]
print(f"{len(questions)} cau truy van co qrels")


In [ ]:
query_embeddings = {}
QBATCH = 8
for start in range(0, len(questions), QBATCH):
    chunk = questions[start:start + QBATCH]
    batch = processor.process_queries([q.query for q in chunk]).to(model.device)
    with torch.no_grad():
        emb = model(**batch).to(torch.float16).cpu()
    for q, e in zip(chunk, emb):
        query_embeddings[q.qid] = e
    if (start // QBATCH) % 5 == 0:
        print(f"  {start + len(chunk)}/{len(questions)}", flush=True)
print(f"done: {len(query_embeddings)} query embeddings")


In [ ]:
import numpy as np

keys = sorted(page_embeddings)  # cot cua ma tran = trang, thu tu co dinh
qids = [q.qid for q in questions]  # hang cua ma tran = truy van

doc_list = [page_embeddings[k].to(torch.float32) for k in keys]
score_matrix = np.zeros((len(qids), len(keys)), dtype=np.float32)

QSCORE_BATCH = 8
for start in range(0, len(qids), QSCORE_BATCH):
    chunk_ids = qids[start:start + QSCORE_BATCH]
    q_list = [query_embeddings[qid].to(torch.float32) for qid in chunk_ids]
    scores = processor.score_multi_vector(q_list, doc_list, batch_size=64, device="cuda")
    score_matrix[start:start + len(chunk_ids)] = scores.cpu().numpy()
    if (start // QSCORE_BATCH) % 5 == 0:
        print(f"  scored {start + len(chunk_ids)}/{len(qids)}", flush=True)

print(f"score matrix: {score_matrix.shape}")


## 7. Kiểm tra nhanh ngay trong Colab

NDCG@10 tự cài đặt đơn giản (không cần `pytrec_eval` ở đây) để có con số ngay, trước khi tải kết
quả về máy local chạy `physics_colqwen_eval.py` (số liệu chính thức, permutation test, fusion
sweep, và độ phủ trên đúng 473 trang bị trượt sẽ được tính ở đó).

In [ ]:
import math

def ndcg10(ranked_keys, gold: dict) -> float:
    dcg = sum((2 ** gold.get(k, 0) - 1) / math.log2(i + 2) for i, k in enumerate(ranked_keys[:10]))
    ideal = sorted(gold.values(), reverse=True)[:10]
    idcg = sum((2 ** g - 1) / math.log2(i + 2) for i, g in enumerate(ideal))
    return 100 * dcg / idcg if idcg > 0 else 0.0

key_index = {k: i for i, k in enumerate(keys)}
scores = []
for row, qid in enumerate(qids):
    gold = qrels[qid]
    ranked = sorted(keys, key=lambda k: -score_matrix[row, key_index[k]])
    scores.append(ndcg10(ranked, gold))
print(f"ColQwen2 visual-only NDCG@10 = {sum(scores) / len(scores):.2f}  (n={len(scores)})")
print("(so sanh: text alpha0.7 = 44.15, CLIP visual-only = 4.45)")


In [ ]:
import json as _json

pool = _json.loads(Path("/content/AXIOM_DE-RD/data/benchmark/vidore_v3/results/physics_served_pool.json").read_text())["queries"]

# Trang gold bi truot han (khong nam trong top-100 cua arm text hien tai)
missed_found = 0
missed_total = 0
for qid in qids:
    if qid not in pool:
        continue
    served_set = set(pool[qid]["candidates"][:100])
    gold = {k for k, v in qrels[qid].items() if v > 0}
    deep = gold - served_set
    if not deep:
        continue
    row = qids.index(qid)
    ranked = sorted(keys, key=lambda k: -score_matrix[row, key_index[k]])
    top10_visual = set(ranked[:10])
    missed_total += len(deep)
    missed_found += len(deep & top10_visual)

print(f"Trong so trang gold bi text trUOT han (khong o top-100 text): {missed_total}")
print(f"ColQwen2 tim lai duoc trong top-10 visual: {missed_found} "
      f"({100 * missed_found / max(missed_total, 1):.1f}%)")


## 8. Xuất kết quả về local

In [ ]:
OUT = Path("/content/physics_colqwen_export")
OUT.mkdir(exist_ok=True)
np.save(OUT / "physics_colqwen_scores.npy", score_matrix)
_json.dump(keys, open(OUT / "physics_colqwen_keys.json", "w"))
_json.dump(qids, open(OUT / "physics_colqwen_qids.json", "w"))

import shutil
shutil.make_archive("/content/physics_colqwen_export", "zip", OUT)
print("exported:", OUT)


In [ ]:
from google.colab import files
files.download("/content/physics_colqwen_export.zip")


## Bước tiếp theo (chạy ở máy local, không cần GPU)

1. Giải nén `physics_colqwen_export.zip` vào `data/work/vidore_physics_colqwen/` trong repo local.
2. Chạy `python research/experiments/physics_colqwen_eval.py` -- script này tính NDCG@10 chính
   thức bằng `pytrec_eval`, permutation test so với baseline text (`physics_sep_test.permutation`),
   sweep trọng số fusion với arm text hiện có, và báo cáo riêng độ phủ trên 473 trang đã xác định
   là bị trượt vì phụ thuộc hình (xem `docs/phan_tich_first_principles.md` §2.3).